# 02 — Leakage-Safe Partial-Cycle Features

Purpose: Build 18 scalar predictors and 64 x 5 sequences from the observable 4.0–3.6 V discharge segment.

Leakage rule: current-cycle full discharged capacity and SOH are labels only.
Preprocessing is fitted on training batteries, validation batteries control
selection and early stopping, and test batteries are scored exactly once.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v3"
RESULTS = ARTIFACT_ROOT / "results"
FIGURES = ARTIFACT_ROOT / "figures"
FEATURES = ARTIFACT_ROOT / "features"
for path in (RESULTS, FIGURES, FEATURES):
    path.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")


Project root: E:\VIT\aiBatteryLifecycle\application


## Target and inputs

SOH = 100 x Q_t / median(Q_1,Q_2,Q_3). Full-cycle Q_t is the target only; inputs use the partial voltage window and prior usage.

In [2]:
import pandas as pd
from scripts.data.build_benchmark_datasets import build_all_benchmarks
summaries = build_all_benchmarks(PROJECT_ROOT / "datasets" / "raw", FEATURES)
pd.DataFrame(summaries)

,dataset,cycles_loaded,cycles_retained,cycles_excluded,batteries,features,sequence_shape
0,NASA,2881,2141,740,30,25,"(2141, 64, 5)"
1,CALCE,3789,3789,0,4,25,"(3789, 64, 5)"
2,Oxford,519,519,0,8,25,"(519, 64, 5)"


In [3]:
from src.utils.config import FEATURE_COLS_V3
forbidden = {"Capacity", "capacity_ah", "SoH", "delta_capacity", "soh_rolling_mean"}
assert not forbidden.intersection(FEATURE_COLS_V3)
print(f"Verified {len(FEATURE_COLS_V3)} predictors with no current-cycle target proxies.")

Verified 18 predictors with no current-cycle target proxies.
